In [1]:
!nvidia-smi

Fri Oct 30 17:28:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Aug_20_01:58:59_PM_PDT_2025
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0


In [5]:
%%writefile vectorAddition.cu
#include <cuda_runtime.h>
#include <iostream>
#include <vector>

__global__ void vector_add(
    const float* A,
    const float* B,
    float* C,
    int N
) {
//calculating global thread ID(tid)
int tid = (blockIdx.x * blockDim.x) + threadIdx.x;

//vector boundary checking
if (tid<N){
    //each thread adds a single element
    C[tid] = A[tid] + B[tid];
}
}

// A, B, C are device pointers (i.e. pointers to memory on the GPU)
//1000 elements, then we could have 1000 GPU threads

extern "C" void solve(
  const float* A,
   const float* B,
   float* C,
   int N
   ) {
    if (N<=0) return;

    int threadsPerBlock = 256;
    int blocksPerGrid = (N + threadsPerBlock - 1) / threadsPerBlock;

    vector_add<<<blocksPerGrid, threadsPerBlock>>>(A, B, C, N);
    cudaDeviceSynchronize();
}
int main(){
  const int N = 1000;

  std::vector<float> A(N, 20.0f);
  std::vector<float> B(N, 23.0f);
  std::vector<float> C(N);

  float *dA, *dB, *dC;

  cudaMalloc((void**)&dA, N* sizeof(float) );
  cudaMalloc((void**)&dB, N* sizeof(float) );
  cudaMalloc((void**)&dC, N* sizeof(float));

  cudaMemcpy(dA, A.data(), N * sizeof(float), cudaMemcpyHostToDevice);
  cudaMemcpy(dB, B.data(), N*sizeof(float), cudaMemcpyHostToDevice);

  solve(dA,dB,dC,N);

  cudaMemcpy(C.data(),dC, N*sizeof(float), cudaMemcpyDeviceToHost);

  std::cout << "C[0] = " <<C[0]<<std::endl;
  std::cout<<"C[999] = "<< C[999]<< std::endl;

  cudaFree(dA);
  cudaFree(dB);
  cudaFree(dC);

  return 0;
}


Overwriting vectorAddition.cu


In [6]:
!nvcc vectorAddition.cu -o vectorAddition
!./vectorAddition

C[0] = 43
C[999] = 43
